In [1]:
import pandas as pd
import mlflow


In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Fraud Detection")

<Experiment: artifact_location='file:C:/Users/trixr/Desktop/FraudDetection/mlflow-data/artifacts/1', creation_time=1781179797941, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781179797941, lifecycle_stage='active', name='Fraud Detection', tags={}, trace_location=None, workspace='default'>

In [3]:
from pathlib import Path

DATA_PATH = Path("./Data")
RAW_DATA_PATH = DATA_PATH / "raw" / "Fraud_Data.csv"

In [4]:
mlflow.start_run(run_name="DataPreProcessing")

<ActiveRun: >

# Load Data with correct Types

In [5]:
df = pd.read_csv(
    RAW_DATA_PATH,
    parse_dates=["signup_time", "purchase_time"]
)

In [6]:
import ipaddress

df["ip_address"] = df["ip_address"].astype("int64").apply(
    lambda x: str(ipaddress.IPv4Address(x))
)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   user_id         151112 non-null  int64         
 1   signup_time     151112 non-null  datetime64[ns]
 2   purchase_time   151112 non-null  datetime64[ns]
 3   purchase_value  151112 non-null  int64         
 4   device_id       151112 non-null  object        
 5   source          151112 non-null  object        
 6   browser         151112 non-null  object        
 7   sex             151112 non-null  object        
 8   age             151112 non-null  int64         
 9   ip_address      151112 non-null  object        
 10  class           151112 non-null  int64         
dtypes: datetime64[ns](2), int64(4), object(5)
memory usage: 12.7+ MB




There were any null or duplicated data found on EDA.
#

In [10]:
import pandas as pd

col = "purchase_value"

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

iqr_outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
# log IQR params
mlflow.log_param("iqr_Q1", Q1)
mlflow.log_param("iqr_Q3", Q3)
mlflow.log_param("iqr_lower_bound", lower_bound)
mlflow.log_param("iqr_upper_bound", upper_bound)

# log IQR metric
mlflow.log_metric("iqr_outlier_count", len(iqr_outliers))

In [12]:
iqr_outliers

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
47,262864,2015-05-03 08:12:33,2015-06-01 08:41:48,91,LYTLRYTPUANEC,Ads,Chrome,F,24,213.101.42.15,0
351,50437,2015-07-12 09:29:46,2015-10-04 13:26:32,107,EOGHGCFLIXVGU,SEO,Chrome,M,18,244.56.219.203,0
470,361782,2015-08-14 20:51:53,2015-09-22 02:53:35,90,BQPWVWUMIDPWI,SEO,Safari,F,21,14.77.100.147,0
661,223039,2015-03-24 14:22:22,2015-07-09 07:58:22,92,LPAUXDPDRMMUK,SEO,FireFox,F,36,12.95.112.112,0
974,370848,2015-07-28 13:26:21,2015-11-17 08:51:28,101,MWJEDXQTARYHR,SEO,IE,M,24,217.110.26.210,1
...,...,...,...,...,...,...,...,...,...,...,...
150535,365551,2015-03-07 23:05:02,2015-04-28 01:35:41,93,YTXYRSFLMAGJM,SEO,Safari,M,41,117.17.38.9,1
150684,216414,2015-01-04 08:30:02,2015-01-04 08:30:03,91,XPSSLODFBMYCV,SEO,IE,F,36,235.217.88.175,1
150715,13743,2015-01-28 10:51:41,2015-02-03 11:05:09,90,BIRDSAJZTDGGL,SEO,IE,F,52,21.111.27.109,0
151066,219307,2015-01-02 02:52:49,2015-04-03 05:52:01,93,JUMGBUSDQKUDK,Ads,Chrome,F,31,18.75.243.153,0


In [11]:
mean = df[col].mean()
std = df[col].std()

sigma_lower = mean - 3 * std
sigma_upper = mean + 3 * std

sigma_outliers = df[(df[col] < sigma_lower) | (df[col] > sigma_upper)]
# log sigma params
mlflow.log_param("sigma_mean", mean)
mlflow.log_param("sigma_std", std)
mlflow.log_param("sigma_lower_bound", sigma_lower)
mlflow.log_param("sigma_upper_bound", sigma_upper)

# log sigma metric
mlflow.log_metric("sigma_outlier_count", len(sigma_outliers))

In [13]:
from collections import defaultdict

# Build a string report
report_lines = []
report_lines.append("DataFrame Summary")
report_lines.append("=" * 50)
report_lines.append(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
report_lines.append("""
\n 
There were any null or duplicated data found on EDA.
There were outliers but the less domain and less clear about what is the unit of values are talking about.
So No actions can be taken for outliers
""")
# Create the final report string
report_str = "\n".join(report_lines)

# Log to MLflow (within an active run)
mlflow.log_text(report_str, "data_preprocessing/dataframe_summary.txt")


In [14]:
mlflow.end_run()

🏃 View run DataPreProcessing at: http://localhost:5000/#/experiments/1/runs/22cf768688a74806a9a694d25016863c
🧪 View experiment at: http://localhost:5000/#/experiments/1
